## How To Use This Notebook

Run top to bottom once, then revisit specific sections for experimentation.

### Section Map

- Section 0: Optional scrape and clean of the NHM 2024 source page
- Section 1: Environment setup (local and Colab notes provided)
- Section 2: Basic RAG progression (no-RAG failure -> RAG success)
- Section 3: Advanced retrieval with reranking
- Section 4: Intent routing with two query tools
- Section 5: Optional tuning sandbox

If you restart the kernel, rerun all cells in order.

In [77]:
# Optional source prep: scrape NHM page and create a tidy local text file
import re
from pathlib import Path

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    %pip install -q requests beautifulsoup4
    import requests
    from bs4 import BeautifulSoup

url = (
    "https://www.nhm.ac.uk/discover/news/2024/december/"
    "dicaprios-snake-saurons-piranha-natural-history-museum-describe-190-new-species-2024.html"
)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Referer": "https://www.nhm.ac.uk/",
}

resp = requests.get(url, headers=headers, timeout=20)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

blocks = soup.find_all(
    "div", class_=lambda c: isinstance(c, str) and c.startswith("_text_")
)

clean = []
for block in blocks:
    text = re.sub(r"\s+", " ", block.get_text(" ", strip=True)).strip()
    if len(text) < 120:
        continue
    if "Receive email updates" in text or "We use cookies" in text:
        continue
    clean.append(text)

if not clean:
    fallback = soup.find("main") or soup
    clean = [re.sub(r"\s+", " ", fallback.get_text(" ", strip=True)).strip()]

output_path = Path("data/new_species_2024.txt")
output_path.parent.mkdir(parents=True, exist_ok=True)
final_text = "\n\n".join(clean)

with output_path.open("w", encoding="utf-8") as f:
    f.write(f"SOURCE_URL: {url}\n\n")
    f.write(final_text)

print(f"Saved {output_path} ({len(final_text)} chars)")
print(final_text[:500] + "...")

Saved data/new_species_2024.txt (6818 chars)
Over the last 12 months our scientists have been describing a dizzying array of new species. From dinosaurs that may have roamed like cattle along Britain’s ancient waterways to tiny diatoms ceaselessly producing oxygen in the warm waters off Australia’s northern beaches. It has been a year of incredible diversity.

Throughout 2024, our scientists have been hard at work describing new species from the depths of the oceans to remote rainforests and even from Welsh living rooms. Not to mention the...


### What This Section Does

In this section you will:
- Install required Python packages
- Detect whether you are in local Jupyter or Colab
- Start Ollama if it is not already running
- Pull one local LLM and one embedding model
- Run a quick smoke test to confirm local inference works

Environment notes:
- Colab-only cells are labeled as COLAB ONLY
- Shared cells are labeled as Shared (Local + Colab)
- Local users can skip COLAB ONLY cells

In [61]:
# Shared (Local + Colab): install Python dependencies for this workshop
%pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [62]:
# Shared (Local + Colab): detect environment and resolve ollama command
import os
import shutil
import sys

IN_COLAB = "google.colab" in sys.modules
print("Environment:", "Colab" if IN_COLAB else "Local")

if IN_COLAB:
    OLLAMA_CMD = shutil.which("ollama") or "ollama"
else:
    candidates = [
        shutil.which("ollama"),
        "/opt/homebrew/bin/ollama",
        "/usr/local/bin/ollama",
        os.path.expanduser("~/.ollama/bin/ollama"),
    ]
    OLLAMA_CMD = next((p for p in candidates if p and os.path.exists(p)), None)

print(f"OLLAMA_CMD: {OLLAMA_CMD}")

Environment: Local
OLLAMA_CMD: /opt/homebrew/bin/ollama


In [63]:
# COLAB ONLY: install ollama inside the runtime (local users can ignore this cell)
import shutil
import subprocess
import sys

IN_COLAB = globals().get("IN_COLAB", "google.colab" in sys.modules)

if IN_COLAB:
    if shutil.which("ollama") is None:
        print("Installing Ollama in Colab runtime...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    OLLAMA_CMD = shutil.which("ollama") or "ollama"
    print(f"Ollama installed at: {OLLAMA_CMD}")
else:
    print("Skipping: this cell is COLAB ONLY.")

Skipping: this cell is COLAB ONLY.


In [64]:
# Shared (Local + Colab): start ollama server if needed
import os
import subprocess
import time
import urllib.request

if not OLLAMA_CMD:
    raise RuntimeError(
        "Ollama command not found. Local users: install Ollama. "
        "Colab users: run the COLAB ONLY install cell first."
    )

os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

def ollama_api_up(url="http://127.0.0.1:11434/api/tags") -> bool:
    try:
        urllib.request.urlopen(url, timeout=2).close()
        return True
    except Exception:
        return False

if not ollama_api_up():
    subprocess.Popen([OLLAMA_CMD, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(12):
        if ollama_api_up():
            break
        time.sleep(1)

if not ollama_api_up():
    raise RuntimeError("Ollama server is not reachable at http://127.0.0.1:11434")

print("Ollama server is running.")
subprocess.run([OLLAMA_CMD, "list"], check=False)

Ollama server is running.
NAME                       ID              SIZE      MODIFIED    
nomic-embed-text:latest    0a109f422b47    274 MB    4 hours ago    
llama3.2:3b                a80c4f17acd5    2.0 GB    4 hours ago    


CompletedProcess(args=['/opt/homebrew/bin/ollama', 'list'], returncode=0)

In [65]:
# Shared (Local + Colab): pull local models (safe to rerun)
import subprocess

LLM_MODEL = "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"

for model_name in [LLM_MODEL, EMBED_MODEL]:
    print(f"Pulling {model_name}...")
    subprocess.run([OLLAMA_CMD, "pull", model_name], check=True)

print(f"Using LLM model: {LLM_MODEL}")
print(f"Using embedding model: {EMBED_MODEL}")

Pulling llama3.2:3b...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ 

Pulling nomic-embed-text...


pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ 

Using LLM model: llama3.2:3b
Using embedding model: nomic-embed-text


pulling manifest ⠼ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 


### Setup Checkpoint

If the smoke test responded correctly, your setup is ready.

If not:
- Re-run Section 1.4 (start service)
- Re-run Section 1.5 (model pull)
- Run `ollama list` in a terminal to verify models are present
- In Colab, restart and rerun Section 1 end-to-end (runtime resets are common)

### RAG Concept In One Minute

RAG has two core steps:
1. Retrieval: find relevant chunks from your documents
2. Generation: answer using those chunks as grounded context

In this section you will see a direct comparison:
- First ask the LLM with no retrieval
- Then ask the same question after indexing the source text

In [66]:
# Configure LlamaIndex to use local Ollama models
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

LLM_MODEL = "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"

Settings.llm = Ollama(model=LLM_MODEL, request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name=EMBED_MODEL)
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print(f"LLM configured: {LLM_MODEL}")
print(f"Embedding model configured: {EMBED_MODEL}")

LLM configured: llama3.2:3b
Embedding model configured: nomic-embed-text


In [67]:
# The same question asked without retrieval context
prompt = "Name me a vegetarian piranha discovered in 2024 and tell me who it was named after."
response = Settings.llm.complete(prompt)

print(f"Query: {prompt}\n")
print("=== Direct LLM response (no RAG) ===")
print(getattr(response, "text", str(response)))

Query: Name me a vegetarian piranha discovered in 2024 and tell me who it was named after.

=== Direct LLM response (no RAG) ===
I couldn't find any information on a vegetarian piranha discovered in 2024 or any other year. Piranhas are carnivorous fish that feed on small animals, including fish, crustaceans, and even carrion. They do not have a known diet that includes plant-based food sources.

If you could provide more context or clarify what you're looking for, I'll be happy to help.


In [68]:
# Shared (Local + Colab): build vector index and rerun the same prompt with retrieval
from pathlib import Path
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

source_file = Path("data/new_species_2024.txt")
if not source_file.exists():
    raise FileNotFoundError(
        "Missing data/new_species_2024.txt. Run Section 0 scraping cell first, "
        "or generate it via scripts/scrape_nhm_new_species.py."
    )

documents = SimpleDirectoryReader(input_files=[str(source_file)]).load_data()
print(f"Loaded {len(documents)} document(s) from {source_file}.")

vector_index = VectorStoreIndex.from_documents(documents)
vector_query_engine = vector_index.as_query_engine(similarity_top_k=3)

# Short aliases used in later sections
index = vector_index
query_engine = vector_query_engine

rag_response = vector_query_engine.query(prompt)
print("=== RAG response ===")
print(rag_response)

Loaded 1 document(s) from data/new_species_2024.txt.
=== RAG response ===
Myloplus sauron. It was named after Sauron from J.R.R. Tolkien's "The Lord of the Rings".


In [69]:
# Inspect which chunks were retrieved for the RAG answer
for i, node in enumerate(rag_response.source_nodes, 1):
    print(f"--- Retrieved Chunk {i} (Score: {node.score:.4f}) ---")
    print(f"{node.text[:250]}...\n")

--- Retrieved Chunk 1 (Score: 0.7089) ---
“This particular group of pterosaurs is known largely from China,” explains Professor Paul Barrett, one of our dinosaur researchers who helped name the flying reptile. “And so, this is a range extension to another part of the world where we didn’t kn...

--- Retrieved Chunk 2 (Score: 0.6891) ---
SOURCE_URL: https://www.nhm.ac.uk/discover/news/2024/december/dicaprios-snake-saurons-piranha-natural-history-museum-describe-190-new-species-2024.html

Over the last 12 months our scientists have been describing a dizzying array of new species. From...

--- Retrieved Chunk 3 (Score: 0.6716) ---
“We have described this remarkable moth, which we know is from central Guyana, but the only specimens and other materials known in the world are from Port Talbot in South Wales.” It’s joined on our new species list by 11 more moths, including two fro...



In [70]:
# Intro RAG: run a diverse new-species question set
sample_questions = [
    "Name me a vegetarian piranha",
    "Which species in the NHM 2024 roundup were named after famous figures, and where were they found?",
    "What made the Carmenta brachyclados discovery in South Wales so unusual?",
    "Summarize the range of organism groups in the 2024 roundup (for example moths, beetles, copepods, amphipods) and why this diversity matters.",
    "How does the article connect new-species discovery to conservation and infrastructure decisions?",
]

for i, q in enumerate(sample_questions, 1):
    print(f"\n=== Question {i} ===")
    print(q)
    response = query_engine.query(q)
    print(response)


=== Question 1 ===
Name me a vegetarian piranha
Myloplus sauron

=== Question 2 ===
Which species in the NHM 2024 roundup were named after famous figures, and where were they found?
Myloplus sauron (piranha), Anguiculus dicaprioi (snake). 
They were found from the Xingu River in Brazil.

=== Question 3 ===
What made the Carmenta brachyclados discovery in South Wales so unusual?
The moth's journey to being discovered in South Wales was an unlikely one. Initially found fluttering around a living room, it later turned out that its caterpillars had gotten stuck to a fragment of seed pod and were carried by the boot of a photographer on an aeroplane before eventually ending up in the UK.

=== Question 4 ===
Summarize the range of organism groups in the 2024 roundup (for example moths, beetles, copepods, amphipods) and why this diversity matters.
The 2024 roundup includes a wide range of diverse organism groups. Insects like moths, beetles, and spiders are represented, along with crustacean

### Why Basic RAG Is Not Always Enough

Vector retrieval can return chunks that share words but are only loosely relevant.
If too much weak context is passed to the LLM, answer quality can drop.

Reranking helps by scoring the retrieved chunks for relevance and keeping only the strongest evidence before synthesis.

In [71]:
# Shared (Local + Colab): baseline retrieval (no reranker)
query = "Which species in the NHM 2024 roundup were named after famous figures, and where were they found?"

baseline_engine = index.as_query_engine(similarity_top_k=8)
baseline_response = baseline_engine.query(query)

print("=== Baseline response ===")
print(baseline_response)

=== Baseline response ===
The Carmenta brachyclados moth was named after its unusual story. One of the scientists involved, Mark Sterling, mentions that a clearwing moth caterpillar got stuck to the boot of a photographer's car and was carried in an aeroplane to the UK. 

Additionally, DiCaprio's snake (Anguiculus dicaprioi) was named after actor and environmentalist Leonard DiCaprio.

Sauron's piranha (Myloplus sauron) was also named after a fictional character, Sauron from The Lord of the Rings.


In [72]:
# Advanced retrieval: LLM-based reranking
from llama_index.core.postprocessor import LLMRerank

RERANK_TOP_N = 1
reranker = LLMRerank(choice_batch_size=4, top_n=RERANK_TOP_N)
rerank_engine = index.as_query_engine(
    similarity_top_k=12,
    node_postprocessors=[reranker],
)
rerank_response = rerank_engine.query(query)

print("=== Reranked response ===")
print(rerank_response)
print(f"Reranked nodes returned: {len(rerank_response.source_nodes)} (top_n={RERANK_TOP_N})")

=== Reranked response ===
The fossil poop called Alococopros milnei was named after AA Milne. It was found from Yorkshire.
Reranked nodes returned: 1 (top_n=1)


### Interpreting The Source Nodes

When comparing baseline vs reranked source nodes, look for:
- Higher relevance to the query intent
- Less off-topic context
- Better support for key claims in the final answer

In this section, one reranked node is expected because `top_n=1` is intentional for teaching clarity.

In [73]:
# Native LlamaIndex routing with tool selection
from llama_index.core import SummaryIndex
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex.from_documents(documents)
summary_query_engine = summary_index.as_query_engine()

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description="Useful for retrieving specific facts, names, or details about individual species discovered in 2024.",
)

summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description="Useful for summarizing the full source, identifying high-level themes, and broad overviews.",
)

router_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[summary_tool, vector_tool],
)

print("Router engine ready.")

Router engine ready.


In [74]:
# Test routing on two different intents
router_queries = [
    "Summarize the main themes of the 2024 new-species discoveries.",
    "What is Myloplus sauron named after?",
]

for q in router_queries:
    print(f"\n--- Query: {q} ---")
    response = router_engine.query(q)
    print("Router selection:", response.metadata.get("selector_result"))
    print("Final answer:")
    print(response)


--- Query: Summarize the main themes of the 2024 new-species discoveries. ---
Router selection: selections=[SingleSelection(index=0, reason='Useful for summarizing the full source, identifying high-level themes, and broad overviews.')]
Final answer:
The year's discoveries highlight the incredible diversity of life on our planet. The vast array of new species described this year underscores the importance of understanding and documenting all species to better inform conservation efforts, policy, and science.

Several key areas are particularly noteworthy, including the deep sea and remote rainforests, where unique and fascinating creatures have been discovered. The descriptions also shed light on previously unexplored ecosystems, such as the Caribbean's golf ball sponges and the remote tepuis of Venezuela.

Fossil discoveries, too, have added significantly to our knowledge, with many specimens coming from Australia's Late Cretaceous rock formations and the Isle of Wight. These finds pr

In [75]:
# Interactive router demo
custom_query = input("Ask a species question or ask for a summary: ").strip()
if custom_query:
    response = router_engine.query(custom_query)
    print("Router selection:", response.metadata.get("selector_result"))
    print("Final answer:")
    print(response)
else:
    print("No query entered.")

Router selection: selections=[SingleSelection(index=1, reason='Useful for retrieving specific facts, names, or details about individual species discovered in 2024.')]
Final answer:
Yes, there was one snake that caught attention. It's named Anguiculus dicaprioi, or DiCaprio's Himalayan snake, after actor and environmentalist Leonard DiCaprio.


## 5) The Tuning Sandbox

RAG is not plug-and-play; it needs tuning.
Use the next cell to change parameters and immediately see how retrieval and answers change.

Things to try:
- Set `CUSTOM_CHUNK_SIZE` to `128` or `1024`
- Set `CUSTOM_TOP_K` to `2`
- Change `TEST_QUERY` to another species question

In [76]:
from llama_index.core.postprocessor import LLMRerank

# ==========================================
# 1. TINKER WITH THESE VARIABLES
# ==========================================
CUSTOM_CHUNK_SIZE = 512
CUSTOM_CHUNK_OVERLAP = 50
CUSTOM_TOP_K = 10         # Initial vector retrieval breadth
CUSTOM_TOP_N_RERANK = 2   # Chunks kept after reranking

TEST_QUERY = "Why was the Carmenta brachyclados discovery in south Wales unusual?"

# ==========================================
# 2. THE RAG PIPELINE (direct LlamaIndex API)
# ==========================================
Settings.chunk_size = CUSTOM_CHUNK_SIZE
Settings.chunk_overlap = CUSTOM_CHUNK_OVERLAP

print("Re-building index with your custom chunk settings...")
sandbox_index = VectorStoreIndex.from_documents(documents)

sandbox_reranker = LLMRerank(choice_batch_size=4, top_n=CUSTOM_TOP_N_RERANK)
sandbox_engine = sandbox_index.as_query_engine(
    similarity_top_k=CUSTOM_TOP_K,
    node_postprocessors=[sandbox_reranker],
)

# ==========================================
# 3. RESULTS
# ==========================================
print(f"\n--- Query: {TEST_QUERY} ---")
sandbox_response = sandbox_engine.query(TEST_QUERY)

print("\nFinal Answer:")
print(sandbox_response)

print(f"\n--- Nodes Used (up to {CUSTOM_TOP_N_RERANK}) ---")
for i, node in enumerate(sandbox_response.source_nodes, 1):
    print(f"\n[Node {i} | Score: {node.score:.4f}]")
    print(f"{node.text[:220]}...")

Re-building index with your custom chunk settings...

--- Query: Why was the Carmenta brachyclados discovery in south Wales unusual? ---

Final Answer:
The Carmenta brachyclados discovery in south Wales was unusual because it started when a moth was found fluttering around a living room, and further investigation revealed that it is actually native to the rainforests of Guyana, South America.

--- Nodes Used (up to 2) ---

[Node 1 | Score: 10.0000]
SOURCE_URL: https://www.nhm.ac.uk/discover/news/2024/december/dicaprios-snake-saurons-piranha-natural-history-museum-describe-190-new-species-2024.html

Over the last 12 months our scientists have been describing a dizzy...

[Node 2 | Score: 9.0000]
“We have described this remarkable moth, which we know is from central Guyana, but the only specimens and other materials known in the world are from Port Talbot in South Wales.” It’s joined on our new species list by 11...
